# Chapter 09: What stays behind

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "09"
__file__ = os.path.abspath("run.py")

## The scene

Wednesday 22 July, 09:30. One call to the statement tool returns Corvane's whole quarter. The
agent needed nine lines of it.

Okafor follows the trace turn by turn. The quarter is there on turn two, where it was asked
for. It is there again on turn three, and on turn nine.

"It is carrying Corvane everywhere it goes," he says.

By the ninth request most of what LANTERN is reading is statements it has finished with. Case
file 02 chapter 20 counted the cost in tokens. Nobody has yet asked what it costs in room.

"Can it put them down?" Varma asks.

## The concept

In an agent's loop every request carries the whole conversation so far. A **tool result**
enters the conversation on the turn that asked for it and stays for every turn after. The
agent reads it once. It pays for it until the run ends.

There are three things you can do with a result the agent has finished with:

- **Clear it.** Replace it with one line that says what it was and that it was cleared. The
  line matters: without it the model acts as if it never looked.
- **Summarise it.** Replace it with a shorter text. Somebody has to write that text, and if
  it is the model, the summary is a new document that can be wrong.
- **Trim it at the tool.** Change the tool so it returns less in the first place: the lines or
  the totals the task needs. What was never returned never has to be put down.

The first two manage what is already in the window. The third decides what enters it.

```mermaid
flowchart TD
    C[Tool call] --> T{{What does the tool return?}}
    T -- every line --> R[Whole statement in the conversation]
    T -- payees, counts, totals --> D[Digest in the conversation]
    R --> U{{After the agent has used it}}
    U -- kept --> K[Carried on every later request]
    U -- cleared --> N[One line: what it was, that it was cleared]
    U -- summarised --> S[A summary the model wrote]
    K & N & S & D --> Q[Next request]
```

## How it works

1. **One call, one quarter.** What `statement(corvane-trading.csv)` returns, and how much of
   it the hop needs.
2. **What the agent carries, request by request.** The checked trace of chapter 08 with a tool
   that returns whole statements. `results.old_share()` measures, before each request, how
   much of the conversation is results from earlier turns.
3. **Three ways to put a result down.** The same trace with old results cleared, with old
   results summarised by the model, and with the tool trimmed to return a digest.
4. **Read the summaries.** Each summary the model wrote, checked in code against the
   statement it summarises.

Clearing is a rewrite of the conversation before each request:

```python
def clear_old(messages):
    out = list(messages)
    for i, call in _old_results(messages):      # every tool result but the newest
        if untouched(out[i]["content"]):
            out[i] = {**out[i], "content": stub(call)}
    return out
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json

from lantern import agent, case, context, learn, results, trail
from lantern.config import ROOT, load_config
from lantern.llm import chat
from lantern.ui import Table, chapter, console, evidence, finding, note, reply, save_shot, step

chapter(
    "09",
    "What stays behind",
    """
Wednesday 22 July, 09:30. One call to the statement tool returns Corvane's whole quarter. The
agent needed nine lines of it.

Okafor follows the trace turn by turn. The quarter is there on turn two, where it was asked
for. It is there again on turn three, and on turn nine.

"It is carrying Corvane everywhere it goes," he says.

By the ninth request most of what LANTERN is reading is statements it has finished with. Case
file 02 chapter 20 counted the cost in tokens. Nobody has yet asked what it costs in room.

"Can it put them down?" Varma asks.
""",
)

MINE = learn.mine()  # --mine: your clear_old() from practice/exercise.py
CLEAR = MINE.clear_old if MINE else results.clear_old
OUT = ROOT / ".scratch" / "09"
RESERVE = 600
MAX_TURNS = 16
# Challenge: which arms run. Remove one to save time, or keep only the one you are changing.
ARMS = ("whole statements, kept", "cleared when used", "summarised when used",
        "trimmed at the tool")
SUMMARISE = ("Summarise this bank statement for a later reader: for every payee the account "
             "paid, give the payee, the number of payments and the total. One line a payee. "
             "Nothing else.")


def trace(tool: str, hook=None):
    """One checked trace. `tool` is "statement" (every line) or "money_out" (a digest)."""
    text, names = trail.listing(True)
    opened: list[str] = []

    def statement(file: str = "", **_):
        if file not in names:
            return {"failed": True, "error": f"No file named {file!r}.", "files": list(names)}
        opened.append(names[file])
        return case.read(f"bank/{names[file]}")

    if tool == "statement":
        schemas = [trail.schema("statement", "Every line of one bank statement, as the bank "
                                "sent it.", file="A file name from the list.")]
        handlers = {"statement": statement}
    else:
        schemas, handlers, opened = trail.tools(names)
    shares = []

    def before(messages):
        messages = hook(messages) if hook else messages
        shares.append(results.old_share(messages, context.count_tokens))
        return messages

    run = agent.run([{"role": "system", "content": trail.RULES + text},
                     {"role": "user", "content": trail.TASK}], chat=chat, schemas=schemas,
                    handlers=handlers, max_turns=MAX_TURNS, max_tokens=RESERVE, limit=limit,
                    until=trail.not_done(opened), before_request=before)
    return {"run": run, "opened": list(dict.fromkeys(opened)), "grade": trail.grade(run.answer),
            "shares": shares}


summaries: list[dict] = []


def write_summary(call: str, content: str) -> str:
    text = chat(content, system=SUMMARISE, seed=7, max_tokens=300).text
    summaries.append({"call": call, "summary": text})
    return text


OUT.mkdir(parents=True, exist_ok=True)
window = load_config()["generation"]["num_ctx"]
limit = window - RESERVE
want = trail.expected()

### 1. One call, one quarter

In [ ]:
step("1. One call, one quarter")
corvane = case.read("bank/corvane-trading.csv")
evidence("statement(corvane-trading.csv): the first four of 37 lines",
         "\n".join(corvane.splitlines()[:5]))
needed = [x for x in corvane.splitlines() if "Lowmarsh" in x]
note(f"The result is {context.count_tokens(corvane):,} tokens. The hop the agent is after is "
     f"on {len(needed)} of its 37 lines, {context.count_tokens(chr(10).join(needed)):,} tokens.")

### 2. What the agent carries, request by request

In [ ]:
step("2. What the agent carries, request by request")
p_share = learn.predict_number(
    "The agent traces the trail with a tool that returns whole statements, and nothing is "
    "ever removed. By its last request, what share of everything it is sent is tool results "
    "from earlier turns?", unit="%")
arms = {}
arms["whole statements, kept"] = trace("statement")
kept = arms["whole statements, kept"]
table = Table(title="Whole statements, kept: every request")
for col in ("request", "tokens sent", "old tool results", "share", "called"):
    table.add_column(col, justify="left" if col == "called" else "right")
for t, (old, total) in zip(kept["run"].turns, kept["shares"], strict=False):
    table.add_row(str(t.n), f"{t.prompt_tokens:,}", f"{old:,}", f"{old / max(total, 1):.0%}",
                  ", ".join(f"{n}({next(iter(a.values()), '')})" for n, a in t.calls)
                  or "(a report)")
console.print(table)
last_old, last_total = kept["shares"][len(kept["run"].turns) - 1]
share = 100 * last_old / last_total
note(f"Stopped: {kept['run'].stopped}. By the last request it sent, {share:.0f}% of the "
     f"conversation was results from earlier turns. Opened {len(kept['opened'])} statements "
     f"and reported {kept['grade']['found']} of {len(want)} hops, totals right on "
     f"{kept['grade']['totals_right']}.")
learn.reveal(p_share, actual=share, tolerance=0.15,
             why="A tool result is asked for once and paid for on every later request. In a "
             "long run the window fills with things the agent has finished with.")

### 3. Three ways to put a result down

In [ ]:
step("3. Three ways to put a result down")
p_best = learn.predict(
    "Three treatments: clear a used result and leave a note; replace it with a summary the "
    "model writes; or change the tool so it returns only who was paid and how much. Which "
    "finds the most hops with the right totals?",
    ["Cleared when used", "Summarised when used", "Trimmed at the tool",
     "None gets a total right"])
if "cleared when used" in ARMS:
    arms["cleared when used"] = trace("statement", CLEAR)
if "summarised when used" in ARMS:
    arms["summarised when used"] = trace("statement", results.summarise_old(write_summary))
if "trimmed at the tool" in ARMS:
    arms["trimmed at the tool"] = trace("money_out")
table = Table(title="The same trace, four treatments of a used result")
for col in ("treatment", "hops found", "totals right", "not hops", "requests",
            "tokens, all requests", "largest request", "opened", "stopped"):
    table.add_column(col)
for name, a in arms.items():
    g, run = a["grade"], a["run"]
    table.add_row(name, f"{g['found']} of {len(want)}", str(g["totals_right"]), str(g["invented"]),
                  str(len(run.turns)), f"{run.total_prompt_tokens:,}", f"{run.largest_request:,}",
                  str(len(a["opened"])), run.stopped.split(":")[0])
console.print(table)
(OUT / "traces.json").write_text(json.dumps(
    {name: {"answer": a["run"].answer, "opened": a["opened"], "grade": a["grade"],
            "stopped": a["run"].stopped,
            "requests": [(t.n, t.prompt_tokens, t.calls) for t in a["run"].turns]}
     for name, a in arms.items()}, indent=2), encoding="utf-8")
(OUT / "summaries.json").write_text(json.dumps(summaries, indent=2), encoding="utf-8")
right = {name: a["grade"]["totals_right"] for name, a in arms.items()}
options = ["cleared when used", "summarised when used", "trimmed at the tool"]
best = max((n for n in options if n in right), key=lambda n: right[n], default=None)
learn.reveal(p_best, answer=3 if not best or right[best] == 0 else options.index(best),
             why="Clearing and summarising manage what is already in the window. Trimming at "
             "the tool decides what enters it, and can do the arithmetic before the model "
             "ever sees a figure.")

### 4. Read the summaries

In [ ]:
step("4. Read the summaries")
checked = 0
if summaries:
    first = summaries[0]
    reply(f"LANTERN's summary of {first['call']}", first["summary"][:500])
    for s in summaries:
        file = s["call"].split("(")[1].rstrip(")")
        real = {p["payee"]: p["total"] for p in trail.digest(file)["paid_out_to"]}
        checked += all(t.replace(".00", "") in s["summary"].replace(",", "") for t in real.values())
    note(f"{len(summaries)} summaries were written. Checked against the statements in code, "
         f"{checked} have every payee's total right. A summary of a tool result is a new "
         "document. The model that wrote it will later read it as if it were the statement.")
else:
    note("The summarising arm did not run.")

cleared = arms.get("cleared when used")
summ = arms.get("summarised when used")
trim = arms.get("trimmed at the tool")


def of(arm, key, default=-1):
    return arm["grade"][key] if arm else default


learn.record(
    hops=len(want),
    statement_tokens=context.count_tokens(corvane),
    kept_found=kept["grade"]["found"],
    kept_totals=kept["grade"]["totals_right"],
    kept_stopped=kept["run"].stopped.split(":")[0],
    kept_requests=len(kept["run"].turns),
    kept_tokens=kept["run"].total_prompt_tokens,
    kept_largest=kept["run"].largest_request,
    kept_old_share_pct=round(share),
    cleared_found=of(cleared, "found"),
    cleared_totals=of(cleared, "totals_right"),
    cleared_largest=cleared["run"].largest_request if cleared else -1,
    cleared_stopped=cleared["run"].stopped.split(":")[0] if cleared else "",
    cleared_opened=len(cleared["opened"]) if cleared else -1,
    summarised_found=of(summ, "found"),
    summarised_totals=of(summ, "totals_right"),
    summarised_stopped=summ["run"].stopped.split(":")[0] if summ else "",
    summaries_written=len(summaries),
    summaries_right=checked,
    trimmed_found=of(trim, "found"),
    trimmed_totals=of(trim, "totals_right"),
    trimmed_tokens=trim["run"].total_prompt_tokens if trim else -1,
    trimmed_largest=trim["run"].largest_request if trim else -1,
    mine=MINE is not None,
)

finding(f"""
One call to the statement tool returns {context.count_tokens(corvane):,} tokens of which the
agent needs {len(needed)} lines. With whole statements kept, the trace stopped
({kept['run'].stopped.split(':')[0]}) after {len(kept['run'].turns)} requests, with {share:.0f}%
of its last request made of results from earlier turns, {kept['grade']['found']} of
{len(want)} hops and {kept['grade']['totals_right']} totals right. Cleared when used:
{of(cleared, 'found')} hops, {of(cleared, 'totals_right')} totals right, largest request
{cleared['run'].largest_request if cleared else 0:,} tokens. Summarised when used:
{of(summ, 'found')} hops, {of(summ, 'totals_right')} totals right, and {checked} of
{len(summaries)} summaries had every total right. Trimmed at the tool, so that it returns
payees, counts and totals: {of(trim, 'found')} of {len(want)} hops and
{of(trim, 'totals_right')} totals right.
""")
save_shot("09-tool-results")

## The nuances

- **Old results were two thirds of the conversation.** By the ninth request, 4,528 of about
  7,000 tokens were statements the agent had already reported from.
- **Kept, the run did not finish.** The tenth request would have been 8,336 tokens against a
  limit of 7,592. The loop stopped and said so. A server would have cut the oldest turns:
  the task itself.
- **Clearing bought room, not accuracy.** The largest request fell from 7,284 tokens to 3,501
  and the agent opened eight statements in place of five. It then opened Arden's, 24,809
  tokens in one result, and no clearing can help a single result larger than the window.
- **No arm that returned whole statements got one total right.** The model does not add
  thirty-seven lines. It never has, in any chapter.
- **The summaries were wrong.** Zero of four had every total right. The first lists Arden
  Logistics as someone Corvane paid, with a total. Arden paid Corvane. The model then read its
  own summary as the statement, and that arm found 2 hops.
- **Trimming at the tool was the only arm that worked:** 7 of 7 with every total right,
  because the tool did the sums and returned nothing else.
- **A cleared result needs its note.** The stub names the call, so the agent can make it
  again. Clearing silently looks, to the model, like never having looked.
- **Rewriting history breaks the cache.** Each request after a clearing starts from a
  different prefix, so the server reads more of it afresh. Chapter 16 measures that.

## Try this

1. Open `.scratch/09/summaries.json` and check each summary against
   `lantern.trail.digest(file)`. Which errors would a reader catch?
2. In the clearing arm, find the request where Arden's statement arrives. What should the
   tool have refused to do?
3. Change the stub so it does not say the result was cleared. Does the agent open the same
   statement again?

## Master it

After this chapter you can measure how much of an agent's context is used tool results,
clear a used result and leave a note that it was cleared, say why a model's summary of a
result is a new document, and decide at the tool what a result should hold.

**Predict, then see.** `make lesson N=09` stops twice: the share of the last request that is
old results, and which treatment gets totals right.

**Build it.** Write `clear_old` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=09          # 7 offline tests
make hint N=09
make lesson N=09 ARGS=--mine
```

**Check yourself.** `make quiz N=09`: seven questions.

**Break it.** `make challenge N=09`: your clearing in the trace; the trimmed tool alone; and a
trace that keeps everything.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 09

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 09

In [ ]:
%run ../../scripts/learn.py hint 09

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 09

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

A statement tool that returns every line puts 2,228 tokens into the conversation for a hop
that is on nine lines, and they stay. By the ninth request two thirds of what LANTERN was
sent was statements it had finished with, and the trace stopped, over budget, with 4 of 7
hops and no total right. Clearing used results halved the largest request and still got no
total right. Summaries written by LANTERN were wrong in four cases of four, and one named a
payer as a payee. Only a tool that returns payees, counts and totals traced all seven hops
with every total right. The agent no longer carries whole statements. It does still carry
every digest and every report, and at some length that too will have to be made smaller.

**Next:** [Chapter 10: Who paid whom](../10-compaction-with-a-schema/README.md)